# Customer Churn Prediction - Model Training

## Overview
This notebook trains multiple machine learning models for churn prediction and compares their performance.

### Goals:
1. Load and preprocess data
2. Train 6 different ML models
3. Evaluate model performance
4. Compare results
5. Select best model
6. Save trained models

## 1. Import Libraries and Set Path

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_loader import load_data
from preprocessing import preprocess_data
from model_trainer import train_models
from evaluation import evaluate_models

# Set styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ All libraries and modules imported successfully')

ModuleNotFoundError: No module named 'pandas'

## 2. Load Dataset

In [2]:
# Load data
df = load_data('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

if df is not None:
    print(f'Dataset shape: {df.shape}')
    print(f'\nFirst rows:')
    df.head()

NameError: name 'load_data' is not defined

## 3. Preprocess Data

In [ ]:
# Preprocess data
preprocessed = preprocess_data(df)

# Extract train-test splits
X_train = preprocessed['X_train']
X_test = preprocessed['X_test']
y_train = preprocessed['y_train']
y_test = preprocessed['y_test']
feature_names = preprocessed['feature_names']

print(f'\n✅ Preprocessing complete!')
print(f'Training set: {X_train.shape}')
print(f'Testing set: {X_test.shape}')
print(f'Features: {len(feature_names)}')

## 4. Train Machine Learning Models

In [ ]:
# Train models
models, trainer = train_models(X_train, y_train, tune=False)

print(f'\n✅ {len(models)} models trained successfully!')
print(f'Models: {list(models.keys())}')

## 5. Evaluate Models

In [ ]:
# Evaluate all models
evaluator, comparison = evaluate_models(models, X_test, y_test)

# Display comparison
print('\nModel Comparison Results:')
print(comparison.to_string())

## 6. Visualize Model Confusion Matrices

In [ ]:
# Plot confusion matrices
fig = evaluator.plot_confusion_matrices(figsize=(15, 10))
plt.tight_layout()
plt.savefig('../visuals/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ Confusion matrices saved')

## 7. Visualize ROC Curves

In [ ]:
# Plot ROC curves
fig = evaluator.plot_roc_curves(figsize=(12, 8))
plt.savefig('../visuals/roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ ROC curves saved')

## 8. Compare Model Metrics

In [ ]:
# Plot metrics comparison
fig = evaluator.plot_metrics_comparison(figsize=(14, 6))
plt.savefig('../visuals/metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ Metrics comparison saved')

## 9. Feature Importance Analysis

In [ ]:
# Get best model
best_model_name = evaluator.get_best_model()
best_model = models[best_model_name]

print(f'🏆 Best Model: {best_model_name}')

# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print(f'\nTop 10 Important Features:')
    print(importance_df.head(10))
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=importance_df.head(10), x='Importance', y='Feature', ax=ax, palette='viridis')
    ax.set_title(f'Top 10 Features - {best_model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../visuals/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print('✅ Feature importance saved')
else:
    print('Feature importance not available for this model type')

## 10. Save Models

In [ ]:
import joblib

# Create models directory
Path('../models').mkdir(exist_ok=True)

# Save all models
for name, model in models.items():
    filename = f"../models/{name.replace(' ', '_').lower()}.pkl"
    joblib.dump(model, filename)
    print(f'✅ {name} saved to {filename}')

# Save preprocessor info
joblib.dump(preprocessed['scaler'], '../models/scaler.pkl')
joblib.dump(feature_names, '../models/feature_names.pkl')
print('\n✅ All models saved successfully!')

## 11. Summary Report

In [ ]:
print('\n' + '='*70)
print('📊 MODEL TRAINING SUMMARY REPORT')
print('='*70)

print(f'\n✅ Models Trained: {len(models)}')
print(f'   {chr(10).join([f"   - {name}" for name in models.keys()])}')

print(f'\n🏆 Best Model: {best_model_name}')
for metric, value in evaluator.results[best_model_name].items():
    if metric not in ['Model', 'Confusion Matrix']:
        if isinstance(value, float):
            print(f'   {metric}: {value:.4f}')

print(f'\n📈 Test Set Performance:')
print(f'   Size: {X_test.shape[0]} samples')
print(f'   Features: {X_test.shape[1]}')

print(f'\n💾 Output Files:')
print(f'   - Models: models/*.pkl')
print(f'   - Visuals: visuals/*.png')
print(f'   - Comparison: comparison_results.csv')

print('\n' + '='*70)